# Multi-class Text Classification Experiment - Experimenting with Different Methods

This notebook will build on the work done in the binary sentiment analysis notebook that experimented with different methods of separating text into 2 categories. Instead this time we will do multi-class classification instead of binary classification. Our data will come from Kaggle and be on news article topics. You can find the dataset [here on Kaggle.](https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset?select=test.csv)

In [1]:
import os
import re
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from IPython.core.magic import register_cell_magic
from sklearn.metrics import f1_score

Matplotlib is building the font cache; this may take a moment.


In [ ]:
# Initialize df to log performance metrics
execution_df = pd.DataFrame(columns=["Cell_Label", "Execution_Time_Sec"])

# This function will be used to log the execution time of each cell in the notebook for which it is run with. 
@register_cell_magic
def log_time(line, cell):
    global execution_df
    # Use the string passed next to the magic command as the cell label
    label = line.strip() if line.strip() else f"Cell_{len(execution_df) + 1}"

    start_time = time.perf_counter()
    shell = get_ipython()
    shell.run_cell(cell)
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    new_row = pd.DataFrame([{"Cell_Label": label, "Execution_Time_Sec": elapsed_time}])
    execution_df = pd.concat([execution_df, new_row], ignore_index=True)


In [ ]:
results_df = pd.DataFrame({}, columns=["Method", "Accuracy", "F1-Micro"])

In [ ]:
train_df = pd.read_csv('train.csv') 
val_df = pd.read_csv('test.csv') 
train_df.head()

In [ ]:
# Saving the mapping of class indices to class names for later use
class_map = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

We see each dataset contains 3 columns, we will concatenate the title to the description.

In [ ]:
train_df["text"] = train_df["Title"] + " - " + train_df["Description"]
val_df["text"] = val_df["Title"] + " - " + val_df["Description"]
train_df = train_df.rename(columns={'Class Index': 'label'})
val_df = val_df.rename(columns={'Class Index': 'label'})

train_df = train_df[["text", "label"]]
val_df = val_df[["text", "label"]]


In [ ]:
train_df.shape

In [ ]:
train_df["label"].value_counts()

In [ ]:
val_df.shape

In [ ]:
val_df["label"].value_counts()

We see the the classes are balanced, so no need to sample.

Lastly we will encode the pos/neg labels as numeric values and we will apply some text cleaning to standardize text. 

In [ ]:
train_df

We will also run some universal text cleaning to standardize the way that text is presented. This is important for more primitive feature collection methods. 

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['text_clean'] = train_df['text'].apply(clean_text)
val_df['text_clean'] = val_df['text'].apply(clean_text)

## Method 1: Bag of Words with Decision Tree
The most primitive way to turn our text into numeric features to use for prediction would be to use a "bag of words" approach where each word in the text becomes a column value and we count the number of times it appears in the sequence. 

In [ ]:
y_train = train_df['label']
y_val = val_df['label']

In [ ]:
%%time
%%log_time Method1_BoWTraining
vectorizer = CountVectorizer(stop_words='english', min_df=2)

X_train = vectorizer.fit_transform(train_df['text_clean'])
X_val = vectorizer.transform(val_df['text_clean'])
bow_model = DecisionTreeClassifier(max_depth=10, random_state=42)
bow_model.fit(X_train, y_train)


In [ ]:
%%time
%%log_time Method1_BoWInference
y_pred = bow_model.predict(X_val)

In [ ]:
print('Validation classification report:')
print(classification_report(y_val, y_pred))

In [ ]:
conf_matrix = confusion_matrix(y_val, y_pred)
acc = bow_model.score(X_val, y_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')
plt.figure(figsize=(9,9))
sns.heatmap(conf_matrix, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
title = 'BOW Model Accuracy Score: {0}'.format(acc)
plt.title(title, size = 15)

In [ ]:

bow_results = pd.DataFrame([{"Method": "Bag of Words", "Accuracy": acc,  "F1-Macro": macro_f1}])
results_df = pd.concat([results_df, bow_results], ignore_index=True)

Results show that using Bag-of-words features can be a viable way to give a model enough content to be intelligent, but there are ways to increase the quality of context we can provide to a model. It seems currently that with this method, the model mistakenly classifies most articles as being about science and technology. 

## Method 2: TF-IDF Vectorizer + Decision Tree
The Bag of Words approach is maybe our most simple method of capturing vocabulary, but we can improve on this. TF-IDF allows our model to understand the importance of terms that show up throughout a dataset. We will see how increasing the complexity of feature collection will affect the accuracy of prediction with the same Decision Tree model. 

In [ ]:
%%time
%%log_time Method2_TFIDFTraining
tfidf = TfidfVectorizer(stop_words='english', min_df=2)
X_train_tfidf = tfidf.fit_transform(train_df['text_clean'])
X_val_tfidf = tfidf.transform(val_df['text_clean'])

tfidf_model = DecisionTreeClassifier(max_depth=10, random_state=42)
tfidf_model.fit(X_train_tfidf, y_train)

In [ ]:
%%time
%%log_time Method2_TFIDFInference
y_pred_tfidf = tfidf_model.predict(X_val_tfidf)

In [ ]:
print('TF-IDF Validation classification report:')
print(classification_report(y_val, y_pred_tfidf))


In [ ]:
conf_matrix = confusion_matrix(y_val, y_pred_tfidf)
acc = tfidf_model.score(X_val_tfidf, y_val)
macro_f1 = f1_score(y_val, y_pred_tfidf, average='macro')
plt.figure(figsize=(9,9))
sns.heatmap(conf_matrix, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
title = 'TF-IDF Model Accuracy Score: {0}'.format(acc)
plt.title(title, size = 15)

In [ ]:
tfidf_results = pd.DataFrame([{"Method": "TF-IDF", "Accuracy": acc,  "F1-Macro": macro_f1}])
results_df = pd.concat([results_df, tfidf_results], ignore_index=True)

With an out of the box TF-IDF vectorizer we actually see a decrease in performance compared to Bag-of-words. There could be greater results with a more complex TF-IDF vectorizer but we will move on to other methods. It still seems currently that with this method, the model mistakenly classifies most articles as being about science and technology. 

## Method 3: Using embeddings as features for a model
Thus far we have seen traditional feature extraction methods but with the rise of generative AI design solutions there has been a rise in using text embeddings for many NLP solutions. We will see how the use of embedding features with no other change in model affects the results. 


In [ ]:
%%time
%%log_time Method3_EmbeddingTraining
# Using sentence-transformers to get embeddings for the tweets
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_train = model.encode(train_df['text_clean'].tolist(), show_progress_bar=True)
embeddings_val = model.encode(val_df['text_clean'].tolist(), show_progress_bar=True)

embedding_log_model = DecisionTreeClassifier(max_depth=10, random_state=42)
embedding_log_model.fit(embeddings_train, y_train)

In [ ]:
print('Train embeddings shape:', getattr(embeddings_train, 'shape', None))
print('Val embeddings shape:', getattr(embeddings_val, 'shape', None))

In [ ]:
%%time
%%log_time Method3_EmbeddingInference
y_pred_embedding = embedding_log_model.predict(embeddings_val)

In [ ]:
print('Embedding Validation classification report:')
print(classification_report(y_val, y_pred_embedding))

In [ ]:
conf_matrix = confusion_matrix(y_val, y_pred_embedding)
acc = embedding_log_model.score(embeddings_val, y_val)
macro_f1 = f1_score(y_val, y_pred_embedding, average='macro')
plt.figure(figsize=(9,9))
sns.heatmap(conf_matrix, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
title = 'Embedding Model Accuracy Score: {0}'.format(acc)
plt.title(title, size = 15)

In [ ]:
embedding_results = pd.DataFrame([{"Method": "Embedding", "Accuracy": acc,  "F1-Macro": macro_f1}])
results_df = pd.concat([results_df, embedding_results], ignore_index=True)

We see a nice improvement here by using text embeddings alone. More complex embedding models may be able to give a classifier even more context and in turn increase the accuracy further. Using text embeddings also seems to have balanced where the model makes errors as it seems to misclassify in a mostly equal way now across classes.

## Method 4: Task Specific Model
This will be a somewhat imperfect comparison because we will test an open source model that can classify news into 40 different topics on our data which contains news with 4 topic labels. However this can still give us a good idea of how a task specific model might work on topic classification. 


In [ ]:
%%time 
%%log_time Method4_TaskSpecificModelAcquisition
task_specific_model = pipeline("text-classification", model="cssupport/bert-news-class")


In [ ]:
%%log_time Method4_TaskSpecificModelInference
y_pred_task_specific = [task_specific_model(text) for text in val_df['text']]

In [ ]:
rows = []
for item in y_pred_task_specific:
    if isinstance(item, list):
        top = item[0] if len(item) > 0 else {}
    elif isinstance(item, dict):
        top = item
    else:
        top = {}
    label = top.get('label') if isinstance(top, dict) else None
    score = top.get('score') if isinstance(top, dict) else None
    rows.append({'label': label, 'score': score})
y_pred_task_specific = pd.DataFrame(rows)
y_pred_task_specific.head()

Some of our categories that the model predicts on are very similar to one another and we can map these to the categories of our labelled data to the best of our ability. 

In [ ]:
def remap_label(label):
    label = label.lower()
    #Original "world" label
    world_labels = ["label_7", "label_23", "label_32", "label_39", "label_40"]
    sports_labels = ["label_27"]
    business_labels = ["label_3"]
    sci_tech_labels = ["label_10", "label_26", "label_31"]
    if label in world_labels:
        return 1
    #Original "sports" label
    elif label in sports_labels:
        return 2
    #Original "business" label
    elif label in business_labels:
        return 3
    #Original "sci/tech" label
    elif label in sci_tech_labels:
        return 4
    else:
        return 0

In [ ]:
y_pred_remapped = y_pred_task_specific["label"].apply(remap_label)

In [ ]:
conf_matrix = confusion_matrix(y_val, y_pred_remapped)
acc = accuracy_score(y_val, y_pred_remapped)
macro_f1 = f1_score(y_val, y_pred_remapped, average='macro')
plt.figure(figsize=(9,9))
sns.heatmap(conf_matrix, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
title = 'Task Specific Model Accuracy Score: {0}'.format(acc)
plt.title(title, size = 15)

In [ ]:
task_specific_results = pd.DataFrame([{"Method": "Task-Specific", "Accuracy": acc,  "F1-Macro": macro_f1}])
results_df = pd.concat([results_df, task_specific_results], ignore_index=True)

Even with the very imperfect mapping of topics from the 40 possible topics of the model to the 4 topics of the dataset, we see an improvement over the previous models. This is a testament to the power of fine tuning a model to a specific task. The comparison of this method would have been further improved if we were able to use a task specific model that only predicted articles as one of the four topics our data contained. Still, given that weakness of mapping this model onto our data, this is a notable improvement. 

## Method 5: Hosting a small local general LLM to predict using a prompt

We might want to test the ability of a general model to solve this problem, however, this will take much longer to run in general. To make this more practical, we will take a sample of tweets to run this on. While this does not result in a 1-to-1 comparison, we can compare it's rate of accuracy. Ultimately this is done purely to make running the code more practical.

In [ ]:
sampled_articles = val_df.sample(500, random_state=1234)

In [ ]:
%%log_time Method5_LLMAcquisition
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()
print("Model loaded!")

In [ ]:
def classify_news(text):
    messages = [
        {"role": "system", "content": "You are a helpful news article classifier. You only respond with one of the following: 'world', 'sports', 'sci/tech', or 'business'."},
        {"role": "user", "content": f"Classify the topic of this news article based on what it is about, either 'world', 'sports', 'sci/tech', or 'business'. Please respond with only the topic name.\n\nArticle: {text}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=5, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
    return 1 if "world" in response else 2 if "sports" in response else 3 if "sci/tech" in response else 4 if "business" in response else 0

In [ ]:
%%log_time Method5_LLMInference
y_pred_llm = [classify_news(article) for article in sampled_articles['text']]

In [ ]:
y_pred_llm.count(1), y_pred_llm.count(2), y_pred_llm.count(3), y_pred_llm.count(4)

In the future we could batch requests or find a better prompt that yields more strict and accurate results. 

In [ ]:
conf_matrix = confusion_matrix(sampled_articles['label'], y_pred_llm)
acc = accuracy_score(sampled_articles['label'], y_pred_llm)
macro_f1 = f1_score(sampled_articles['label'], y_pred_llm, average='macro')
plt.figure(figsize=(9,9))
sns.heatmap(conf_matrix, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
title = 'LLM Topic Accuracy Score: {0}'.format(acc)
plt.title(title, size = 15)

In [ ]:
llm_results = pd.DataFrame([{"Method": "LLM", "Accuracy": acc,  "F1-Macro": macro_f1}])
results_df = pd.concat([results_df, llm_results], ignore_index=True)

## Results and Conclusions

In [ ]:
execution_df

In [ ]:
results_df

In [ ]:
ax = results_df.plot(x='Method', y=['Accuracy', 'F1-Macro'], kind='bar', rot=45, figsize=(12, 8))
ax.grid(axis='y', color='gray', linestyle='--', linewidth=0.5) 
plt.title('Results of Multi-Class Text Classification Methods')
plt.ylabel('Value of Metric')
plt.show()

In [ ]:
def classify_phase(label):
    s = str(label).lower()
    if 'infer' in s:
        return 'Inference'
    return 'Training/Acquisition'

def extract_method(label):
    label = str(label)
    cleaned = re.sub(r'(?i)(training|train|inference|infer|acquisition|acquisition|acquisi|load|download)', '', label)
    cleaned = cleaned.replace('_', ' ').strip()
    return cleaned if cleaned else label

execution_df['Phase'] = execution_df['Cell_Label'].apply(classify_phase)
execution_df['Method'] = execution_df['Cell_Label'].apply(extract_method)
execution_df['Execution_Time_Sec'] = execution_df['Execution_Time_Sec'].astype(float)

summary = execution_df.groupby(['Method','Phase'])['Execution_Time_Sec'].sum().unstack(fill_value=0)
display(summary)

ax = summary.plot(kind='bar', stacked=True, figsize=(12,6))
ax.set_ylabel('Time (seconds)')
ax.set_title('Execution time of Training and Inference by Method')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()